## choice

In [1]:
VERSION = "1.1.3"
PERCENTAGE = 100
STYLE = "pisco_euclid"
SPLIT = "dataset"

## TRAIN/VAL DATASETS FOR NN

## TEST DATASETS FOR NN & S++
# "2.*.*": "lhs + g round(3) + 10 gal/patch + g<=|0.06| + e gauss + seeds 22 + airy",
# "2.0.1": "1_000",
# "2.0.2": "2_000 (1000 x 2 : shape noise cancellation with add noise)",
# "2.0.3": "4_000 (1000 x 4 : shape noise cancellation with add noise + sub noise)",
# "2.0.4": "1_000 with lhs_pos",
# "2.0.5": "1_000 with pixel out",
# "2.1.1": "25_000",
# "2.1.2": "50_000 (25_000 x 2 : shape noise cancellation with add noise)",
# "2.1.3": "100_000 (25_000 x 4 : shape noise cancellation with add noise + sub noise)",
# "2.1.4": "25_000 with lhs_pos",
# "2.1.5": "25_000 with pixel out",
# "2.2.0": "100_000",

## libs

In [ ]:
import resource

import hiplot
import holoviews
import numpy
import panel
import tensorflow_datasets as tfds
from astropy.stats import mad_std
from functions_cdfs import CDF_HLR_WRT_MAG, CDF_MAG, CDF_NS_WRT_MAG, get_inv_cdfs
from functions_images import build_psf
from functions_lhs import draw_lhs, fit_g_vs_z, generate_shear_scaled, redshift_vs_shear_data
from pandas import DataFrame, concat, read_csv
from utils import (
    get_indices,
    get_pairs,
    getting_norms,
    save_psf,
    scale,
    DIR_DATASETS,
    slice_csv,
    DIRS,
)
from plots import (
    before_after,
    cdf_inv_cdf,
    cdf_inv_cdf_case,
    cdf_inv_cdf_mag_case,
    check_image,
    check_psf,
    combined_plot,
    heatmaps_inputs,
    histo_galaxies,
    histo_images,
    images_psf,
    interactive_dist,
    interactive_hist,
    interactive_pairplot,
    large_ellps,
    lhs_visu,
    lhs_visu_single,
    noise_shape,
    plot_example,
    style_norma,
    visu_inputs_ouputs,
)

holoviews.extension("bokeh", "plotly")
panel.extension()

resource.setrlimit(resource.RLIMIT_NOFILE, (50000, 50000))

DIR_VERSION = DIRS["pisco_euclid"] / VERSION
DIR_SE = DIRS["pisco_euclid"] / f"{VERSION}_sextractor"
DIR_IMAGES = DIRS["images"] / VERSION
DIR_IMAGES.mkdir(parents=True, exist_ok=True)

## load all params

In [ ]:
dataset, ds_info = tfds.load(
    f"{STYLE}:{VERSION}", split=f"{SPLIT}[:{PERCENTAGE}%]", data_dir=DIR_DATASETS, with_info=True
)
print(ds_info.metadata)
print(dataset.element_spec.keys())

In [ ]:
TOTAL_ELEMENTS = dataset.cardinality().numpy()
hypercube_img = slice_csv(DIR_VERSION / "images_parameters_true.csv", TOTAL_ELEMENTS)
hypercube_gal = slice_csv(DIR_VERSION / "galaxies_parameters_true.csv", TOTAL_ELEMENTS * 10)

keys_dataset = ["image"]
if TOTAL_ELEMENTS <= 50_000:
    keys_dataset.append("image_psf")

all_params_dataset = dataset.map(lambda x: {key: x[key] for key in keys_dataset})
all_params = list(all_params_dataset.as_numpy_iterator())

images = numpy.stack([item["image"] for item in all_params])
if TOTAL_ELEMENTS <= 50_000:
    image_psf = numpy.stack([item["image_psf"] for item in all_params])
    image_psf = image_psf[0]

del all_params_dataset, all_params, dataset

# compute stats

In [ ]:
if not (DIR_VERSION / "all_statistics.csv").exists():
    DataFrame(
        [
            {
                "min_images": numpy.amin(images),
                "max_images": numpy.amax(images),
                "mean_images": numpy.mean(images),
                "std_images": mad_std(images),
                "min_avg_g": numpy.amin(hypercube_img[["avg_g1", "avg_g2"]].to_numpy()),
                "max_avg_g": numpy.amax(hypercube_img[["avg_g1", "avg_g2"]].to_numpy()),
                "min_avg_obs_e": numpy.amin(hypercube_img[["avg_obs_e1", "avg_obs_e2"]].to_numpy()),
                "max_avg_obs_e": numpy.amax(hypercube_img[["avg_obs_e1", "avg_obs_e2"]].to_numpy()),
                "min_avg_int_e": numpy.amin(hypercube_img[["avg_int_e1", "avg_int_e2"]].to_numpy()),
                "max_avg_int_e": numpy.amax(hypercube_img[["avg_int_e1", "avg_int_e2"]].to_numpy()),
                "min_obs_e": numpy.amin(hypercube_gal[["obs_e1", "obs_e2"]].to_numpy()),
                "max_obs_e": numpy.amax(hypercube_gal[["obs_e1", "obs_e2"]].to_numpy()),
                "min_int_e": numpy.amin(hypercube_gal[["int_e1", "int_e2"]].to_numpy()),
                "max_int_e": numpy.amax(hypercube_gal[["int_e1", "int_e2"]].to_numpy()),
            }
        ]
    ).to_csv(DIR_VERSION / "all_statistics.csv", index=False)

# shape noise cancellation

In [ ]:
grouped_values, duplicate_pairs_image = get_indices(hypercube_gal)

In [ ]:
CHOICE = 5
apply_arcsinh = True

if len(duplicate_pairs_image) != 0:
    indices = duplicate_pairs_image[CHOICE]

    noise_shape(
        DIR_IMAGES, images[numpy.array(indices)], hypercube_img.iloc[indices], apply_arcsinh
    )
    del indices

del apply_arcsinh, CHOICE

# psf

In [ ]:
if TOTAL_ELEMENTS <= 50_000:
    images_psf(DIR_IMAGES, image_psf)

# histo hypercubes

In [ ]:
histo_images(hypercube_img, DIR_IMAGES)

In [ ]:
histo_galaxies(hypercube_gal, dir_plots=DIR_IMAGES)

# params

In [ ]:
pairs_img = [("avg_g1", "avg_g2"), ("avg_obs_e1", "avg_obs_e2"), ("avg_int_e1", "avg_int_e2")]
pairs_gal = [
    ("int_e1", "int_e2"),
    ("obs_e1", "obs_e2"),
    ("x", "y"),
    ("hlr", "mag"),
    ("hlr", "sersic"),
]
all_pairs = get_pairs(pairs_img, hypercube_img, pairs_gal, hypercube_gal)

In [ ]:
alpha = 0.3
size = 2
lhs_visu(DIR_IMAGES, all_pairs, alpha, size)

In [ ]:
NB = 1
alpha = 0.5

lhs_visu_single(DIR_IMAGES, all_pairs[NB], alpha)

# pairplots

In [ ]:
size_dots = 2
columns_drop_img = ["seeds", "hashed_key", "k"]
interactive_pairplot(
    hypercube_img.drop(columns=columns_drop_img, inplace=False),
    size_dots,
    DIR_IMAGES / "pp_images",
    width=1250,
    height=1250,
)

In [ ]:
size_dots = 2
columns_drop_gal = ["int_e1", "int_e2"]
interactive_pairplot(
    hypercube_gal.drop(columns=columns_drop_gal, inplace=False),
    size_dots,
    DIR_IMAGES / "pp_gal_params",
    width=1350,
    height=1350,
)

# examples

In [ ]:
print(selected_number := numpy.random.permutation(images.shape[0])[:3])
# print(selected_number := numpy.array([6, 8]))

selected_images = images[selected_number]
plot_example(selected_images, DIR_IMAGES)
del selected_number, selected_images

In [ ]:
loggy = True
visu_inputs_ouputs(dbb={"inputs": hypercube_img, "simu": images}, loggy=loggy)

# large avg ellipticities

In [ ]:
WHICH = "obs"
# WHICH = "int"
THRESH_1 = 0.2
THRESH_2 = 0.4

conditions = (abs(hypercube_img[f"avg_{WHICH}_e1"]) > THRESH_1) & (
    abs(hypercube_img[f"avg_{WHICH}_e2"]) < THRESH_2
)

filtered_inputs = hypercube_img[conditions]
print("cases possible", index_numbers := filtered_inputs.index.tolist())

if len(index_numbers) != 0:
    select_numbers = numpy.random.permutation(numpy.array(index_numbers))[:3]
    large_ellps(
        selected_images=images[select_numbers],
        elps1=hypercube_img[f"avg_{WHICH}_e1"][select_numbers],
        elps2=hypercube_img[f"avg_{WHICH}_e2"][select_numbers],
        number=select_numbers,
        directory=DIR_IMAGES,
        name=f"avg_{WHICH}",
    )

    del select_numbers, filtered_inputs, conditions, index_numbers, THRESH_1, THRESH_2, WHICH

# large single ellipticities

In [ ]:
WHICH = "obs"
# WHICH = "int"
THRESH_1 = 0.5
THRESH_2 = 0.2

conditions = (abs(hypercube_gal[f"{WHICH}_e1"]) > THRESH_1) & (
    abs(hypercube_gal[f"{WHICH}_e2"]) < THRESH_2
)

filtered_inputs = hypercube_gal[conditions]
print("cases possible", index_numbers := filtered_inputs.index.tolist())

if len(index_numbers) != 0:
    select_number_e = numpy.random.permutation(numpy.array(index_numbers))[:3]
    large_ellps(
        selected_images=images[select_number_e // 10],
        elps1=hypercube_gal[f"{WHICH}_e1"][select_number_e],
        elps2=hypercube_gal[f"{WHICH}_e2"][select_number_e],
        number=select_number_e // 10,
        directory=DIR_IMAGES,
        name=f"single_{WHICH}",
    )

    del THRESH_1, THRESH_2, WHICH, conditions, filtered_inputs, select_number_e

# norms outputs

In [ ]:
images_norms = getting_norms(data=images)
panel.Tabs(
    ("HISTO", interactive_hist(data=images_norms)), ("DIST", interactive_dist(data=images_norms))
)

# parallel plots

In [ ]:
columns_drop = ["seeds", "hashed_key", "k"]

_ = hiplot.Experiment.from_dataframe(
    dataframe=concat(
        (images_norms, hypercube_img.drop(columns=columns_drop, inplace=False)), axis=1
    )
).display(force_full_width=True)

# heatmaps

In [ ]:
columns_drop_img = ["seeds", "hashed_key", "k"]
columns_drop_gal = ["int_e1", "int_e2"]

heatmaps_inputs(
    hypercube_gal.drop(columns=columns_drop_gal, inplace=False),
    concat((images_norms, hypercube_img.drop(columns=columns_drop_img, inplace=False)), axis=1),
    DIR_IMAGES,
)

# psf sextractor

In [ ]:
all_psf = list(DIRS["psf"].glob("*.fits"))

for each_psf in all_psf:
    print(each_psf)

CHOICE = 1
print("CHOICE", all_psf[CHOICE])
check_psf(all_psf[CHOICE])

In [ ]:
style = "airy"  #  airy / euclid
psf = build_psf(style)
image_size = 81  # must have odd size for S++
pixel_scale = 0.06

save_psf(DIRS["psf"], style, psf=psf, image_size=image_size, pixel_scale=pixel_scale)
del style, psf, image_size, pixel_scale

# scaling

In [ ]:
sigma = read_csv(DIR_VERSION / "all_statistics.csv")["std_images"][0]

style_norma(images.flatten(), style="regular", directory=DIR_IMAGES)

In [ ]:
style_norma(
    numpy.arcsinh(images.flatten() / (sigma * 3)), style="arcsinh-sig-3", directory=DIR_IMAGES
)

In [ ]:
NUM = 0

before_after(images[NUM, :, :], DIR_IMAGES)
del NUM

# cdfs

In [ ]:
inv_cdf_mag, inv_cdfs_ns, inv_cdfs_hlr, cases = get_inv_cdfs()

In [ ]:
cdf_inv_cdf_mag_case(inv_cdf_mag, CDF_MAG, DIR_IMAGES / "proba_mag")

In [ ]:
cdf_inv_cdf(inv_cdfs_ns, CDF_NS_WRT_MAG, cases, DIR_IMAGES / "proba_ns")
cdf_inv_cdf(inv_cdfs_hlr, CDF_HLR_WRT_MAG, cases, DIR_IMAGES / "proba_hlr")

In [ ]:
CASE = 21.0
DEG = 4

# fmt: off
cdf_inv_cdf_case(
    # CASE, inv_cdfs_hlr[CASE], CDF_HLR_WRT_MAG, DIR_IMAGES / "proba_case_hlr", deg=DEG
    CASE, inv_cdfs_ns[CASE], CDF_NS_WRT_MAG, DIR_IMAGES / "proba_case_ns", deg=DEG
)
del DEG, CASE

# reshift VS shear


In [ ]:
hypercube = scale(
    array=draw_lhs(2, seed=22222, n_samples=1000),
    low=-0.05,
    high=0.05,
)
g_magnitude = numpy.sqrt(hypercube[:, 0] ** 2 + hypercube[:, 1] ** 2)

z_grid = numpy.linspace(0.2, 5, 10000)

z_pdf = z_grid**2 * numpy.exp(-z_grid / 0.8)

z_data, g_data = redshift_vs_shear_data()

deg = 1
shunsheng_fn = fit_g_vs_z(g_data, z_data, deg=deg)

z, g1_scaled, g2_scaled = generate_shear_scaled(
    hypercube[:, 0], hypercube[:, 1], shunsheng_fn, z_grid, z_pdf, g_magnitude
)

combined_plot(
    DIR_IMAGES, z, z_grid, z_pdf, z_data, g_data, shunsheng_fn, hypercube, g1_scaled, g2_scaled
)

In [ ]:
# from plots import images_redshift
# CHOICE = 5
# images_redshift(DIR_IMAGES, images[CHOICE], images_z_pixel[CHOICE], images_z_flux[CHOICE])

# check assoc vs image (if sextractor database created)


In [ ]:
CHOICE = "000010"
compress = True

check_image(DIR_SE / f"{CHOICE}_simu", DIR_IMAGES, compress)
del CHOICE, compress